## 1. **Import Libraries**

In [ ]:
from pathlib import Path
import json
import random
import os
import logging

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.applications import EfficientNetV2S

logging.getLogger('tensorflow').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')

gpus = tf.config.list_physical_devices('GPU')
has_gpu = len(gpus) > 0
print("TensorFlow version:", tf.__version__)
print("GPU Available      :", has_gpu)
if not has_gpu:
    print("No GPU detected — training on CPU (may be slow). Consider reducing epochs.")



## 2. **Data Loading and Preprocessing**

In [ ]:
# Data will be loaded from local processed directory


In [ ]:
# ── Paths (local) ─────────────────────────────────────────────────────────────────
import os
_nb_dir = Path(os.getcwd())
if _nb_dir.name == 'notebooks':
    PROJECT_ROOT = _nb_dir.parent
else:
    PROJECT_ROOT = _nb_dir
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_DIR = PROCESSED_DATA_DIR / "train"
VAL_DIR   = PROCESSED_DATA_DIR / "val"
TEST_DIR  = PROCESSED_DATA_DIR / "test"

MODEL_DIR  = PROJECT_ROOT / "models"
HEATMAP_DIR = PROJECT_ROOT / "tmp" / "heatmaps"

for d in [MODEL_DIR, HEATMAP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hyper-parameters ──────────────────────────────────────────────────────
IMG_SIZE   = (224, 224)
IMG_HEIGHT = IMG_SIZE[0]
IMG_WIDTH  = IMG_SIZE[1]
CHANNELS   = 3

HEATMAP_SIZE  = 48     # heatmap head output resolution
HEATMAP_SIGMA = 10     # Gaussian sigma in IMG_SIZE space (scaled to HEATMAP_SIZE)

BATCH_SIZE    = 32
EPOCHS_PHASE1 = 10     # head-only warm-up
EPOCHS_PHASE2 = 60     # full fine-tune
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Train path : {TRAIN_DIR}")
print(f"Val path   : {VAL_DIR}")
print(f"Test path  : {TEST_DIR}")



In [ ]:
required_dirs = {"train": TRAIN_DIR, "validation": VAL_DIR, "test": TEST_DIR}
for split_name, dir_path in required_dirs.items():
    if not dir_path.exists():
        raise FileNotFoundError(f"Khong tim thay folder {split_name}: {dir_path}")
print("Tat ca folder train/val/test da ton tai.")


In [ ]:
def count_images_by_class(directory):
    image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]
    records = []
    for class_dir in sorted(directory.iterdir()):
        if class_dir.is_dir():
            image_count = sum(len(list(class_dir.glob(ext))) for ext in image_extensions)
            records.append({"class_name": class_dir.name, "image_count": image_count})
    return pd.DataFrame(records)

train_count_df = count_images_by_class(TRAIN_DIR)
val_count_df   = count_images_by_class(VAL_DIR)
test_count_df  = count_images_by_class(TEST_DIR)

print("TRAIN DATASET");      print(train_count_df.to_string())
print("VALIDATION DATASET"); print(val_count_df.to_string())
print("TEST DATASET");       print(test_count_df.to_string())
print("Total train images      :", train_count_df["image_count"].sum())
print("Total validation images :", val_count_df["image_count"].sum())
print("Total test images       :", test_count_df["image_count"].sum())


In [ ]:
def plot_class_distribution(df, title):
    plt.figure(figsize=(14, 5))
    plt.bar(df["class_name"], df["image_count"])
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

plot_class_distribution(train_count_df, "Train Dataset Distribution")
plot_class_distribution(val_count_df,   "Validation Dataset Distribution")
plot_class_distribution(test_count_df,  "Test Dataset Distribution")


In [ ]:
# EfficientNetV2S expects pixel values in [0, 255].
# Its include_preprocessing=True handles normalisation internally.
# We use preprocessing_function from tf.keras.applications.efficientnet_v2
# for the ImageDataGenerator pipeline.

train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.12,
    height_shift_range=0.12,
    zoom_range=0.12,
    horizontal_flip=True,
    vertical_flip=False,           # MRI brain scans are typically upright
    brightness_range=[0.8, 1.2],   # simulate MRI contrast variation
    fill_mode="nearest",
    preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet_v2.preprocess_input
)

train_data = train_datagen.flow_from_directory(
    directory=TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_data = val_test_datagen.flow_from_directory(
    directory=VAL_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_data = val_test_datagen.flow_from_directory(
    directory=TEST_DIR,
    target_size=IMG_SIZE,
    color_mode="rgb",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_data.num_classes
CLASS_NAMES  = list(train_data.class_indices.keys())
print("Number of classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)


In [ ]:
images, labels = next(train_data)

plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    # De-normalise for display (EfficientNetV2 preprocess scales to [-1, 1])
    img = (images[i] + 1.0) / 2.0
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    class_index = np.argmax(labels[i])
    plt.title(CLASS_NAMES[class_index], fontsize=8)
    plt.axis("off")

plt.tight_layout()
plt.show()

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)


### Compute class weights (handle class imbalance)

Brain-tumour datasets are often skewed (rare subtypes have fewer MRI scans).
Weighted loss ensures the model pays equal attention to every tumour type,
which is crucial for clinical utility and for achieving high macro-F1.


In [ ]:
train_labels = train_data.classes   # integer class indices for all training images

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)
class_weights = {i: float(w) for i, w in enumerate(class_weights_array)}

print("Class weights (first 5):", dict(list(class_weights.items())[:5]))
print("Max weight :", max(class_weights.values()))
print("Min weight :", min(class_weights.values()))

# ── Custom weighted loss (nhung class_weights vao loss fn — Keras 3 compatible) ──
_cw_tensor = tf.constant(
    [class_weights[i] for i in range(NUM_CLASSES)], dtype=tf.float32
)  # shape [NUM_CLASSES]

def weighted_crossentropy(y_true, y_pred):
    """CategoricalCrossentropy voi label_smoothing=0.1 va class weights nhung vao."""
    ls = 0.1
    num_cls = tf.cast(tf.shape(y_true)[-1], tf.float32)
    y_smooth = y_true * (1.0 - ls) + ls / num_cls
    per_sample = -tf.reduce_sum(y_smooth * tf.math.log(y_pred + 1e-7), axis=-1)  # [B]
    true_idx    = tf.argmax(y_true, axis=-1)                  # [B]
    sample_wt   = tf.gather(_cw_tensor, true_idx)             # [B]
    return tf.reduce_mean(per_sample * sample_wt)

print("Custom weighted loss function created.")


### Offline Heatmap Pre-computation (từ folder structure — không cần DATA.json)

**Chiến lược tạo heatmap không cần JSON:**
- **Normal classes** (`class_name.startswith("Normal")`): heatmap = **mảng toàn số 0**
  → không có tổn thương thật, tránh data leakage hoàn toàn.
- **Tumor classes**: heatmap = **Gaussian rộng ở trung tâm ảnh**
  → không có tọa độ chính xác từ JSON, nhưng Gaussian rộng (σ lớn) vẫn
  dạy model chú ý vào vùng trung tâm — nơi khối u thường xuất hiện trong MRI.
  Heatmap đóng vai trò *attention regulariser* nhẹ (loss_weight = 0.1).

**Tại sao pre-compute thay vì on-the-fly?**
Tránh CPU bottleneck trong vòng lặp training — heatmap được tính một lần và lưu `.npy`.


In [ ]:
# ── Gaussian heatmap generator ───────────────────────────────────────────
def make_gaussian_heatmap(hm_size: int, class_name: str, sigma: float) -> np.ndarray:
    """
    Tạo heatmap (hm_size × hm_size, float32):
      - Normal* classes → mảng toàn 0  (DATA LEAKAGE PREVENTION)
      - Tumor  classes  → Gaussian rộng ở trung tâm
    Không cần tọa độ JSON — chỉ dùng tên class để phân biệt Normal vs Tumor.
    """
    heatmap = np.zeros((hm_size, hm_size), dtype=np.float32)

    if class_name.startswith("Normal"):
        return heatmap   # Normal → blank: không có tổn thương thật

    # Tumor → Gaussian tại trung tâm heatmap
    cx = cy = hm_size / 2.0
    x = np.arange(hm_size, dtype=np.float64)
    y = np.arange(hm_size, dtype=np.float64)[:, np.newaxis]
    heatmap = np.exp(
        -4.0 * np.log(2) * ((x - cx)**2 + (y - cy)**2) / (sigma**2)
    )
    return heatmap.astype(np.float32)


# ── Scan folder → danh sách ảnh với class name ────────────────────────────
def scan_split_dir(root_dir: Path, class_to_idx: dict) -> list:
    """
    Duyệt root_dir/ClassName/*.{jpg,png,...}
    Trả về list[dict] với img_path và class name.
    Thứ tự giống flow_from_directory (sorted by class, sorted by filename).
    """
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    items = []
    for cls_dir in sorted(root_dir.iterdir()):
        if not cls_dir.is_dir() or cls_dir.name not in class_to_idx:
            continue
        for img_path in sorted(cls_dir.iterdir()):
            if img_path.suffix.lower() in exts:
                items.append({'img_path': img_path, 'class': cls_dir.name})
    return items


class_to_idx = train_data.class_indices   # lấy từ flow_from_directory

train_items = scan_split_dir(TRAIN_DIR, class_to_idx)
val_items   = scan_split_dir(VAL_DIR,   class_to_idx)
test_items  = scan_split_dir(TEST_DIR,  class_to_idx)

print(f"Train items : {len(train_items):,}")
print(f"Val   items : {len(val_items):,}")
print(f"Test  items : {len(test_items):,}")


In [ ]:
# ── Pre-compute heatmaps và lưu .npy ──────────────────────────────────────
def precompute_heatmaps(items: list, hm_dir: Path,
                         hm_size: int, sigma: float,
                         split_name: str) -> list:
    """
    Với mỗi ảnh trong items:
      → tạo heatmap theo class name (Normal → zeros, Tumor → Gaussian)
      → lưu ra hm_dir/{split_name}_{i:05d}.npy
    Trả về list đường dẫn .npy tương ứng 1-1 với items.
    """
    paths = []
    sigma_hm = sigma * hm_size / max(IMG_SIZE)  # scale sigma sang không gian hm

    for i, item in enumerate(items):
        out_path = hm_dir / f"{split_name}_{i:05d}.npy"
        if not out_path.exists():
            hm = make_gaussian_heatmap(hm_size, item['class'], sigma_hm)
            np.save(out_path, hm)
        paths.append(str(out_path))

    # In thống kê
    n_normal = sum(1 for it in items if it['class'].startswith('Normal'))
    n_tumor  = len(items) - n_normal
    print(f"[{split_name:5s}] {len(items):5,} heatmaps — "
          f"Normal (zeros): {n_normal:,}  |  Tumor (Gaussian): {n_tumor:,}")
    return paths


train_hm_paths = precompute_heatmaps(train_items, HEATMAP_DIR,
                                      HEATMAP_SIZE, HEATMAP_SIGMA, "train")
val_hm_paths   = precompute_heatmaps(val_items,   HEATMAP_DIR,
                                      HEATMAP_SIZE, HEATMAP_SIGMA, "val")
test_hm_paths  = precompute_heatmaps(test_items,  HEATMAP_DIR,
                                      HEATMAP_SIZE, HEATMAP_SIGMA, "test")

total_hm = len(train_hm_paths) + len(val_hm_paths) + len(test_hm_paths)
print(f"\nTotal heatmaps pre-computed: {total_hm:,}")
print(f"Saved to: {HEATMAP_DIR}")


In [ ]:
# ── Visualise heatmap samples ────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
sample_indices = random.sample(range(len(train_items)), 8)

for col, idx in enumerate(sample_indices):
    item = train_items[idx]
    img  = cv2.cvtColor(cv2.imread(str(item['img_path'])), cv2.COLOR_BGR2RGB)
    img  = cv2.resize(img, (160, 160))
    hm   = np.load(train_hm_paths[idx])
    hm_r = cv2.resize(hm, (160, 160))

    axes[0, col].imshow(img)
    axes[0, col].imshow(hm_r, cmap='hot', alpha=0.5, vmin=0, vmax=1)
    axes[0, col].set_title(item['class'], fontsize=5)
    axes[0, col].axis('off')

    axes[1, col].imshow(hm_r, cmap='hot', vmin=0, vmax=1)
    axes[1, col].set_title(
        'ZERO (Normal)' if item['class'].startswith('Normal') else 'Gaussian',
        fontsize=5)
    axes[1, col].axis('off')

plt.suptitle('Row 1: Image + Heatmap Overlay  |  Row 2: Target Heatmap\n'
             'Normal → blank zeros  |  Tumor → center Gaussian',
             fontsize=9)
plt.tight_layout()
plt.show()


### Dual-Head Data Generator

`ImageDataGenerator.flow_from_directory` chỉ yield `(image, label)` — không hỗ trợ dual output.
`DualHeadGenerator` đọc ảnh từ **cùng thư mục** (`TRAIN_DIR / VAL_DIR / TEST_DIR`),
dùng **cùng preprocessing** như notebook gốc (`efficientnet_v2.preprocess_input`),
và yield `(image, {'class_output': one_hot_label, 'heatmap_output': heatmap})`.


In [ ]:
class DualHeadGenerator(tf.keras.utils.Sequence):
    """
    Custom Sequence generator — đọc ảnh từ thư mục split đã có sẵn.
    Yield: (images_batch, {'class_output': labels, 'heatmap_output': heatmaps})

    Tham số augmentation giống hệt ImageDataGenerator trong notebook gốc:
      rotation 20°, shift 0.12, zoom 0.12, horizontal_flip, brightness [0.8,1.2]
    """

    def __init__(self, items: list, hm_paths: list,
                 class_to_idx: dict, num_classes: int,
                 img_size: tuple, hm_size: int,
                 batch_size: int, augment: bool = False,
                 shuffle: bool = True):
        self.items       = items
        self.hm_paths    = hm_paths
        self.c2i         = class_to_idx
        self.num_classes = num_classes
        self.img_h, self.img_w = img_size
        self.hm_size     = hm_size
        self.batch_size  = batch_size
        self.augment     = augment
        self.shuffle     = shuffle
        self.preprocess  = tf.keras.applications.efficientnet_v2.preprocess_input
        self.on_epoch_end()

    # ── Augmentation helpers (cùng tham số notebook gốc) ──────────────────
    def _augment(self, img: np.ndarray) -> np.ndarray:
        """Apply augmentation identical to notebook-04 ImageDataGenerator."""
        h, w = img.shape[:2]

        # Horizontal flip (p=0.5)
        if random.random() < 0.5:
            img = cv2.flip(img, 1)

        # Random rotation ±20°
        angle = random.uniform(-20, 20)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h),
                             borderMode=cv2.BORDER_REFLECT_101)

        # Random shift ±12%
        tx = random.uniform(-0.12, 0.12) * w
        ty = random.uniform(-0.12, 0.12) * h
        M2 = np.float32([[1, 0, tx], [0, 1, ty]])
        img = cv2.warpAffine(img, M2, (w, h),
                              borderMode=cv2.BORDER_REFLECT_101)

        # Random zoom ±12%  (crop + resize)
        scale = random.uniform(1 - 0.12, 1 + 0.12)
        new_h, new_w = int(h * scale), int(w * scale)
        img_r = cv2.resize(img, (new_w, new_h))
        if scale >= 1.0:
            y0 = (new_h - h) // 2
            x0 = (new_w - w) // 2
            img = img_r[y0:y0+h, x0:x0+w]
        else:
            pad_y = (h - new_h) // 2
            pad_x = (w - new_w) // 2
            img = cv2.copyMakeBorder(img_r, pad_y, h-new_h-pad_y,
                                      pad_x, w-new_w-pad_x,
                                      cv2.BORDER_REFLECT_101)

        # Random brightness [0.8, 1.2]
        factor = random.uniform(0.8, 1.2)
        img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

        return img

    def __len__(self):
        return int(np.floor(len(self.items) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]

        X     = np.empty((self.batch_size, self.img_h, self.img_w, 3), np.float32)
        y_cls = np.empty((self.batch_size, self.num_classes),           np.float32)
        y_hm  = np.empty((self.batch_size, self.hm_size, self.hm_size, 1), np.float32)

        for i, k in enumerate(batch_idx):
            item = self.items[k]

            # Load image (BGR → RGB → resize)
            img = cv2.imread(str(item['img_path']))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_w, self.img_h))

            # Augmentation (train only)
            if self.augment:
                img = self._augment(img)

            # EfficientNetV2 preprocessing ([0,255] → [-1,1])
            X[i] = self.preprocess(img.astype(np.float32))

            # One-hot label
            y_cls[i] = tf.keras.utils.to_categorical(
                self.c2i[item['class']], self.num_classes)

            # Pre-computed heatmap
            hm = np.load(self.hm_paths[k])
            y_hm[i] = hm[..., np.newaxis]

        return X, {'class_output': y_cls, 'heatmap_output': y_hm}

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.items))
        if self.shuffle:
            np.random.shuffle(self.indexes)


# ── Khởi tạo generators ──────────────────────────────────────────────────
train_gen = DualHeadGenerator(
    train_items, train_hm_paths, class_to_idx, NUM_CLASSES,
    IMG_SIZE, HEATMAP_SIZE, BATCH_SIZE, augment=True,  shuffle=True)

val_gen   = DualHeadGenerator(
    val_items,   val_hm_paths,   class_to_idx, NUM_CLASSES,
    IMG_SIZE, HEATMAP_SIZE, BATCH_SIZE, augment=False, shuffle=False)

test_gen  = DualHeadGenerator(
    test_items,  test_hm_paths,  class_to_idx, NUM_CLASSES,
    IMG_SIZE, HEATMAP_SIZE, BATCH_SIZE, augment=False, shuffle=False)

print(f"Train batches : {len(train_gen)}")
print(f"Val   batches : {len(val_gen)}")
print(f"Test  batches : {len(test_gen)}")

# Sanity check
X_s, Y_s = train_gen[0]
print(f"\nBatch image shape  : {X_s.shape}  dtype={X_s.dtype}")
print(f"Batch label shape  : {Y_s['class_output'].shape}")
print(f"Batch heatmap shape: {Y_s['heatmap_output'].shape}")
print(f"Pixel range        : [{X_s.min():.2f}, {X_s.max():.2f}]")


## **Build Model – Dual-Head EfficientNetV2S**

### Architecture rationale

| Component | Choice | Why |
|-----------|--------|-----|
| Backbone | EfficientNetV2S (ImageNet) | SOTA accuracy-per-parameter; fused-MBConv blocks train fast |
| Head 1 (Classification) | GAP → BN → Dense(512, swish) → Dropout(0.4) → Dense(256, swish) → Dropout(0.3) → Dense(30, softmax) | Giữ nguyên head notebook gốc — đã proven hiệu quả |
| Head 2 (Heatmap) | Conv2DTranspose ×3 → sigmoid 48×48 | Attention regulariser nhẹ — hướng dẫn model chú ý vùng tổn thương |
| Loss | CE label_smoothing=0.1 (weight 1.0) + MSE heatmap (weight 0.1) | Heatmap chỉ đóng vai trò phụ, không lấn át classification |
| Optimiser phase 1 | Adam lr=1e-3 | Fast convergence for head-only training |
| Optimiser phase 2 | Adam + CosineDecayRestarts lr=2e-4 → 1e-6 | Smooth annealing avoids sharp overfitting during fine-tune |
| Regularisation | Dropout 0.4/0.3, L2(1e-4) on Dense, class weights | Multi-level defence against overfitting |

### Two-phase training strategy
1. **Phase 1 – Warm-up (10 epochs)**: Backbone frozen. Only the custom heads are trained.
2. **Phase 2 – Fine-tune (up to 60 epochs)**: Top 80 layers of backbone unfrozen with very low LR.


In [ ]:
from tensorflow.keras import regularizers

def build_dual_head_model(input_shape, heatmap_size, num_classes):
    """
    Dual-Head EfficientNetV2S:
      Head 1 — Classification : GAP → BN → Dense(512) → Dense(256) → Dense(30, softmax)
      Head 2 — Heatmap decoder: Conv2DTranspose ×3 → sigmoid (heatmap_size × heatmap_size)
    """
    # ── Backbone ──────────────────────────────────────────────────────────
    base_model = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
        include_preprocessing=True    # built-in normalisation layer
    )
    base_model.trainable = False      # freeze during phase 1

    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    x = base_model(inputs, training=False)   # (batch, 7, 7, 1280) for 224×224

    # ── Head 1: Classification (giống hệt notebook gốc) ───────────────────
    gap = layers.GlobalAveragePooling2D(name="gap")(x)
    gap = layers.BatchNormalization(name="bn_head")(gap)

    gap = layers.Dense(
        512, activation="swish",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_512"
    )(gap)
    gap = layers.Dropout(0.4, name="drop_1")(gap)

    gap = layers.Dense(
        256, activation="swish",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_256"
    )(gap)
    gap = layers.Dropout(0.3, name="drop_2")(gap)

    class_out = layers.Dense(
        num_classes, activation="softmax",
        dtype="float32",              # force float32 — mixed-precision stability
        name="class_output"
    )(gap)

    # ── Head 2: Heatmap Decoder (7×7 → 14×14 → 28×28 → 56×56 → crop 48×48)
    h = layers.Conv2DTranspose(
        128, (3, 3), strides=2, padding="same",
        activation="relu", name="up_1")(x)      # 14×14
    h = layers.Conv2DTranspose(
        64,  (3, 3), strides=2, padding="same",
        activation="relu", name="up_2")(h)      # 28×28
    h = layers.Conv2DTranspose(
        1,   (3, 3), strides=2, padding="same",
        activation="sigmoid",
        dtype="float32",
        name="heatmap_raw")(h)                  # 56×56

    # Crop to exact heatmap_size × heatmap_size
    crop = (56 - heatmap_size) // 2
    heatmap_out = layers.Cropping2D(
        ((crop, crop), (crop, crop)),
        name="heatmap_output")(h)               # 48×48

    model = tf.keras.Model(
        inputs=inputs,
        outputs=[class_out, heatmap_out],
        name="BrainTumor_DualHead_EfficientNetV2S"
    )
    return model, base_model


input_shape = (IMG_HEIGHT, IMG_WIDTH, CHANNELS)
model, base_model = build_dual_head_model(input_shape, HEATMAP_SIZE, NUM_CLASSES)
model.summary(line_length=100, expand_nested=False)

total     = model.count_params()
trainable = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"\nTotal params     : {total:,}")
print(f"Trainable params : {trainable:,}  (heads only — backbone frozen)")


In [ ]:
# ── Shared loss + compile helper (dùng cho cả 2 phase) ───────────────────
# weighted_crossentropy đã được định nghĩa ở trên (nhúng class_weights)

METRICS_CLS = [
    "accuracy",
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
    tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy"),
]

def compile_model(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss={
            "class_output"  : weighted_crossentropy,  # class-balanced CE
            "heatmap_output": "mean_squared_error",
        },
        loss_weights={
            "class_output"  : 1.0,   # primary task — full weight
            "heatmap_output": 0.1,   # attention regulariser — light weight
        },
        metrics={
            "class_output"  : METRICS_CLS,
            "heatmap_output": [],
        }
    )

compile_model(model, lr=1e-3)
print("Model compiled — Phase 1 ready.")


## **Phase 1 – Head Warm-up (backbone frozen)**

In [ ]:
best_model_path  = MODEL_DIR / "brain_tumor_best.keras"
final_model_path = MODEL_DIR / "brain_tumor_final.keras"

callbacks_phase1 = [
    ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_class_output_accuracy",
        save_best_only=True, mode="max", verbose=1),
    EarlyStopping(
        monitor="val_class_output_loss",
        patience=5, restore_best_weights=True, verbose=1),
]

print("=== PHASE 1: Training head only ===")
history_phase1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1,
)


## **Phase 2 – Fine-tuning (unfreeze top 80 backbone layers)**

In [ ]:
# Unfreeze top 80 layers of EfficientNetV2S
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 80

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Keep BatchNorm in inference mode — prevents fine-tune instability
for layer in base_model.layers[fine_tune_at:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"Trainable backbone layers: {trainable_count} / {len(base_model.layers)}")

# Cosine decay with warm restarts – smooth LR, prevents sharp overfitting
total_steps = len(train_gen) * EPOCHS_PHASE2
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=2e-4,
    first_decay_steps=total_steps // 3,
    t_mul=1.0,
    m_mul=0.9,
    alpha=1e-6
)

compile_model(model, lr=lr_schedule)

callbacks_phase2 = [
    ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_class_output_accuracy",
        save_best_only=True, mode="max", verbose=1),
    EarlyStopping(
        monitor="val_class_output_loss",
        patience=10, restore_best_weights=True, verbose=1),
]

print("=== PHASE 2: Fine-tuning top 80 backbone layers ===")
history_phase2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_phase2,
)

model.save(final_model_path)
print(f"Final model saved: {final_model_path}")


## **Plot Training History**

In [ ]:
def merge_histories(h1, h2):
    """Merge Phase-1 and Phase-2 history dicts, mapping dual-head keys → display keys."""
    # Map dual-head metric names → display names (compatible with notebook-04 plot code)
    key_pairs = [
        ("class_output_accuracy",     "accuracy"),
        ("val_class_output_accuracy", "val_accuracy"),
        ("class_output_loss",         "loss"),
        ("val_class_output_loss",     "val_loss"),
    ]
    merged = {}
    for src_key, dst_key in key_pairs:
        v1 = h1.history.get(src_key, [])
        v2 = h2.history.get(src_key, [])
        merged[dst_key] = v1 + v2
    return merged

merged     = merge_histories(history_phase1, history_phase2)
phase1_end = len(history_phase1.history["class_output_accuracy"])


def plot_training_history(merged, phase1_end):
    acc      = merged["accuracy"]
    val_acc  = merged["val_accuracy"]
    loss     = merged["loss"]
    val_loss = merged["val_loss"]
    epochs_range = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, acc,     label="Train Accuracy")
    axes[0].plot(epochs_range, val_acc, label="Val Accuracy")
    axes[0].axvline(phase1_end, color="gray", linestyle="--", label="Phase 1 -> 2")
    axes[0].set_title("EfficientNetV2S DualHead – Accuracy Curve")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs_range, loss,     label="Train Loss")
    axes[1].plot(epochs_range, val_loss, label="Val Loss")
    axes[1].axvline(phase1_end, color="gray", linestyle="--", label="Phase 1 -> 2")
    axes[1].set_title("EfficientNetV2S DualHead – Loss Curve")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.show()


plot_training_history(merged, phase1_end)


## **Evaluation & Metrics**

In [ ]:
# Load best checkpoint for evaluation
best_model = tf.keras.models.load_model(best_model_path)

# evaluate returns [total_loss, class_loss, heatmap_loss, accuracy, ...]
eval_results = best_model.evaluate(test_gen, verbose=1)
print(f"Test Loss (class) : {eval_results[1]:.4f}")
print(f"Test Accuracy     : {eval_results[3]:.4f}")


In [ ]:
# Predict on test set — collect class probs, heatmaps, raw images
cls_probs_all, heatmaps_all, raw_imgs_all = [], [], []

for X_batch, _ in test_gen:
    preds = best_model(X_batch, training=False)
    cls_probs_all.append(preds[0].numpy())
    heatmaps_all.append(preds[1].numpy())
    raw_imgs_all.append(X_batch)

y_pred_prob = np.concatenate(cls_probs_all, axis=0)
y_heatmaps  = np.concatenate(heatmaps_all,  axis=0)
raw_imgs    = np.concatenate(raw_imgs_all,   axis=0)

y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.array([class_to_idx[item['class']]
                   for item in test_items[:len(y_pred)]])

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)


In [ ]:
accuracy           = accuracy_score(y_true, y_pred)
precision_macro    = precision_score(y_true, y_pred, average="macro",    zero_division=0)
recall_macro       = recall_score(y_true,    y_pred, average="macro",    zero_division=0)
f1_macro           = f1_score(y_true,        y_pred, average="macro",    zero_division=0)
precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted    = recall_score(y_true,    y_pred, average="weighted", zero_division=0)
f1_weighted        = f1_score(y_true,        y_pred, average="weighted", zero_division=0)

metrics_summary = {
    "model_name"        : "EfficientNetV2S DualHead (fine-tuned)",
    "image_size"        : str(IMG_SIZE),
    "heatmap_size"      : HEATMAP_SIZE,
    "batch_size"        : BATCH_SIZE,
    "epochs_phase1"     : len(history_phase1.history["class_output_loss"]),
    "epochs_phase2"     : len(history_phase2.history["class_output_loss"]),
    "test_accuracy"     : float(accuracy),
    "precision_macro"   : float(precision_macro),
    "recall_macro"      : float(recall_macro),
    "f1_macro"          : float(f1_macro),
    "precision_weighted": float(precision_weighted),
    "recall_weighted"   : float(recall_weighted),
    "f1_weighted"       : float(f1_weighted),
}

metrics_df = pd.DataFrame([metrics_summary])
print(metrics_df.to_string())
# metrics_df saved to CSV skipped (no log files)


In [ ]:
print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
# Confusion matrix – counts and normalised side by side
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Confusion Matrix (counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Confusion Matrix (normalised)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# 3×3 Prediction Grid: original | heatmap overlay | heatmap alone
n_grid = 9
fig, axes = plt.subplots(3, n_grid, figsize=(24, 8))

for col in range(n_grid):
    # De-normalise image (EfficientNetV2 preprocess: [-1,1] → [0,1])
    img_disp = (raw_imgs[col] + 1.0) / 2.0
    img_disp = np.clip(img_disp, 0, 1)

    hm      = y_heatmaps[col, :, :, 0]
    hm_big  = cv2.resize(hm, (IMG_WIDTH, IMG_HEIGHT))

    pred_cls = CLASS_NAMES[y_pred[col]]
    true_cls = CLASS_NAMES[y_true[col]]
    correct  = y_pred[col] == y_true[col]
    color    = "green" if correct else "red"

    # Row 0: original image + true label
    axes[0, col].imshow(img_disp)
    axes[0, col].set_title(f"True:\n{true_cls}", fontsize=6)
    axes[0, col].axis("off")
    for sp in axes[0, col].spines.values():
        sp.set_edgecolor(color); sp.set_linewidth(3)
    axes[0, col].set_frame_on(True)

    # Row 1: overlay + predicted label
    axes[1, col].imshow(img_disp)
    axes[1, col].imshow(hm_big, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    axes[1, col].set_title(f"Pred:\n{pred_cls}", fontsize=6, color=color)
    axes[1, col].axis("off")

    # Row 2: heatmap alone
    axes[2, col].imshow(hm_big, cmap="hot", vmin=0, vmax=1)
    axes[2, col].set_title("Attention Map", fontsize=6)
    axes[2, col].axis("off")

row_labels = ["Original Image", "Heatmap Overlay", "Predicted Heatmap"]
for row, lbl in enumerate(row_labels):
    axes[row, 0].set_ylabel(lbl, fontsize=9, rotation=90, labelpad=40)

plt.suptitle("Prediction Grid  |  Green border = Correct  |  Red border = Wrong",
             fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
print("=" * 65)
print("  TRAINING & EVALUATION COMPLETE")
print("=" * 65)
print(f"  Best model       : {best_model_path}")
print(f"  Test Accuracy    : {accuracy*100:.2f}%")
print(f"  Macro F1-score   : {f1_macro:.4f}")
print(f"  Weighted F1      : {f1_weighted:.4f}")
print(f"  Model saved to   : {MODEL_DIR}")
print("=" * 65)
